# Infinite Training
> **Goal-based training that runs until the model is good enough — not until a fixed epoch count.**

Unlike standard training (fixed epochs over a fixed dataset), infinite training:

- Maintains a **NWW pool** — the larger the pool, the harder the negatives it can find.
- Every epoch, infers on a random `scan_size` subset → keeps only **hard negatives** (those the current model scores ≥ 0.5).
- Optionally synthesises fresh **voice-converted positives** per epoch via chatterbox-onnx.
- Stops when all goals are met *and* F1 stops improving — or when a hard plateau is detected (no new hard negatives for N epochs).

This notebook covers:
1. **Smoke test** — validate the loop works end-to-end in a few minutes
2. **Standard run** — train until real F1/EER targets are met
3. **With VC synthesis** — grow the positive pool each epoch via voice conversion
4. **NWW pool size ablation** — show how pool size affects hard-negative quality
5. **Per-epoch dashboard** — F1, EER, hard-neg count, mining cache growth

---

## How stopping works

```
while True:
    mine hard negatives from NWW pool (scan_size files)
    [optionally] synthesise vc_per_epoch new positives
    train one epoch
    evaluate on test set → check goals

    stop if:
      max_epochs reached                          (hard ceiling)
      OR hard_plateau epochs with no new HN       (pool exhausted)
      OR all goals met AND patience_after epochs
         of no improvement                        (converged)
```

Key stopping parameters:

| Parameter | Default | Meaning |
|-----------|---------|--------|
| `target_f1` | 0.92 | Minimum F1 required before patience counts |
| `target_eer` | 0.08 | Maximum EER required |
| `patience_after` | 10 | Epochs of no improvement *after* goals are met |
| `hard_plateau` | 20 | Epochs with zero new hard negatives → stop (pool exhausted) |
| `min_epochs` | 20 | Never stop before this |
| `max_epochs` | None | Hard ceiling; use for smoke tests |

---

## Prerequisites

A dataset must already exist at `experiments/<wake_word>/dataset/`. Run the quickstart first:

```bash
.venv/bin/python scripts/train/train_hey_mycroft.py
# or
ww_trainer-quickstart --wake-word "hey mycroft" --output-dir ./experiments/hey_mycroft
```

Or use Cell 4 in this notebook to generate the dataset inline.

---

## Platform notes

| Platform | Recommended settings |
|----------|-----------------------|
| **Kaggle GPU T4** | `DEVICE=cuda`, `SCAN_SIZE=5000`, `BATCH_SIZE=32` |
| **Local CPU** | `SCAN_SIZE=200`, `MAX_EPOCHS=10`, `BATCH_SIZE=8` for smoke test |
| **Colab** | Same as Kaggle; set `OUTPUT_DIR=/content/drive/MyDrive/ww_output` |

---

## Outputs

```
OUTPUT_DIR/
├── best_f1.pt               # best checkpoint by F1
├── best_f1.onnx             # classifier head (inference)
├── best_f1_featurizer.onnx  # featurizer (inference) — required!
├── final.pt                 # last epoch checkpoint
├── metrics.csv              # per-epoch F1 / EER / loss
├── mining_cache.pt          # hardness scores — persisted across runs
└── vc_epoch_positives/      # VC-generated positives (if vc_per_epoch > 0)
```

## Cell 2 — Configuration

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD   = os.environ.get("WAKE_WORD",   "hey mycroft")
OUTPUT_DIR  = os.environ.get("OUTPUT_DIR",  "./ww_output")
DEVICE      = os.environ.get("DEVICE",      "auto")
SEED        = int(os.environ.get("SEED",    "42"))

# ── Model ─────────────────────────────────────────────────────────────────────
ARCH        = os.environ.get("ARCH",        "gru")   # gru | ffn | cnn | bcresnet
LOSS        = os.environ.get("LOSS",        "bce")   # bce | focal | rppl
HIDDEN_DIM  = int(os.environ.get("HIDDEN_DIM", "128"))
BATCH_SIZE  = int(os.environ.get("BATCH_SIZE", "16"))
LR          = float(os.environ.get("LR",    "5e-4"))

# ── Stopping goals ────────────────────────────────────────────────────────────
TARGET_F1       = float(os.environ.get("TARGET_F1",       "0.92"))
TARGET_EER      = float(os.environ.get("TARGET_EER",      "0.08"))
PATIENCE_AFTER  = int(os.environ.get("PATIENCE_AFTER",    "10"))
HARD_PLATEAU    = int(os.environ.get("HARD_PLATEAU",      "20"))
MIN_EPOCHS      = int(os.environ.get("MIN_EPOCHS",        "20"))
MAX_EPOCHS      = os.environ.get("MAX_EPOCHS",  "")    # empty = unlimited
MAX_EPOCHS      = int(MAX_EPOCHS) if MAX_EPOCHS.strip() else None

# ── Mining ────────────────────────────────────────────────────────────────────
SCAN_SIZE       = int(os.environ.get("SCAN_SIZE",     "5000"))
NEG_THRESHOLD   = float(os.environ.get("NEG_THRESHOLD", "0.5"))
NEG_MULTIPLIER  = float(os.environ.get("NEG_MULTIPLIER", "3.0"))

# ── Voice conversion (optional) ───────────────────────────────────────────────
VC_PER_EPOCH    = int(os.environ.get("VC_PER_EPOCH",  "0"))   # 0 = disabled
VC_BACKEND      = os.environ.get("VC_BACKEND",        "chatterbox-onnx")

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE       = int(os.environ.get("N_POSITIVE",    "500"))
LANG             = os.environ.get("LANG",              "en")
DOWNLOAD_AUGMENT = os.environ.get("DOWNLOAD_AUGMENT",  "true").lower() == "true"
CUSTOM_TRAIN_CSV = os.environ.get("CUSTOM_TRAIN_CSV",  "")
CUSTOM_TEST_CSV  = os.environ.get("CUSTOM_TEST_CSV",   "")

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_URI    = os.environ.get("MLFLOW_URI",    "")
MLFLOW_SECRET = os.environ.get("MLFLOW_SECRET", "MLFLOW_TOKEN")

# ── Derived paths ─────────────────────────────────────────────────────────────
from pathlib import Path
_ww_slug = WAKE_WORD.lower().replace(" ", "_")
EXPERIMENT_DIR = Path(OUTPUT_DIR) / _ww_slug
DATASET_DIR    = EXPERIMENT_DIR / "dataset"
MODEL_DIR      = EXPERIMENT_DIR / "models" / f"{ARCH}_{LOSS}_h{HIDDEN_DIM}_infinite"

print(f"Wake word : {WAKE_WORD!r}")
print(f"Goals     : F1≥{TARGET_F1}  EER≤{TARGET_EER}  min_epochs={MIN_EPOCHS}  max_epochs={MAX_EPOCHS}")
print(f"Mining    : scan_size={SCAN_SIZE}  neg_threshold={NEG_THRESHOLD}  neg_multiplier={NEG_MULTIPLIER}")
print(f"VC        : {VC_PER_EPOCH} positives/epoch via {VC_BACKEND!r}" if VC_PER_EPOCH else "VC: disabled")
print(f"Model dir : {MODEL_DIR}")

## Cell 3 — Install & platform detection

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "onnx", "onnxruntime", "tqdm", "psutil")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)
print(f"Platform: {_platform} | Device: {DEVICE} | Wake word: {WAKE_WORD!r}")

## Cell 4 — MLflow setup

In [ ]:
import os

if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["MLFLOW_TRACKING_TOKEN"] = UserSecretsClient().get_secret(MLFLOW_SECRET)
        print(f"MLflow token injected from Kaggle Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"No Kaggle secret found — MLflow disabled. ({e})")

if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    print(f"MLflow URI: {MLFLOW_URI}")
else:
    print("MLFLOW_URI not set — tracking disabled")

## Cell 5 — Dataset

Loads an existing dataset or generates one via TTS. The NWW pool is built from the CSV negatives plus any `not_wake_word_*.wav` files found under `dataset/negatives/`.

The larger the NWW pool, the more hard negatives the miner can find each epoch. A pool of 500 CSV negatives is enough for smoke testing; for production, use `scripts/data/download_hdd4_datasets.py` to get a large corpus (100K+).

In [ ]:
import shutil, csv as _csv, random
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, f"Only {free_gb:.1f} GB free — need at least 3 GB."
print(f"Disk free: {free_gb:.1f} GB")

if CUSTOM_TRAIN_CSV:
    # BYO mode
    from ww_trainer.utils import read_dataset_csv
    train_rows = read_dataset_csv(Path(CUSTOM_TRAIN_CSV))
    if CUSTOM_TEST_CSV:
        test_rows = read_dataset_csv(Path(CUSTOM_TEST_CSV))
    else:
        random.seed(SEED)
        random.shuffle(train_rows)
        cut = int(len(train_rows) * 0.8)
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        for path, rows in [(split_dir/"train.csv", train_rows[:cut]),
                           (split_dir/"test.csv",  train_rows[cut:])]:
            if not path.exists():
                with open(path, "w", newline="") as f:
                    _csv.writer(f).writerows(rows)
        train_rows = read_dataset_csv(split_dir/"train.csv")
        test_rows  = read_dataset_csv(split_dir/"test.csv")
    print(f"BYO: {len(train_rows)} train / {len(test_rows)} test")
    _aug_dirs = {}
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    from ww_trainer.utils import read_dataset_csv

    _train_csv = DATASET_DIR / "train" / "metadata.csv"
    if _train_csv.exists():
        print(f"Reusing existing dataset at {DATASET_DIR}")
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=_train_csv,
            test_csv=DATASET_DIR / "test" / "metadata.csv",
            positives_dir=DATASET_DIR / slug / "positives",
            negatives_dir=DATASET_DIR / slug / "negatives",
            bg_noise_dir=DATASET_DIR / "augmentation" / "bg_noise",
            music_dir=DATASET_DIR / "augmentation" / "music",
            rir_dir=DATASET_DIR / "augmentation" / "rir",
        )
    else:
        print(f"Generating dataset for '{WAKE_WORD}'...")
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD, output_dir=DATASET_DIR,
            n_positive=N_POSITIVE, lang=LANG,
            adversarial=True, vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT, seed=SEED,
        ))

    train_rows = read_dataset_csv(_dr.train_csv)
    test_rows  = read_dataset_csv(_dr.test_csv)
    _aug_dirs  = {}
    if hasattr(_dr, "bg_noise_dir") and _dr.bg_noise_dir and Path(_dr.bg_noise_dir).exists():
        _aug_dirs["bg_noise_folder"] = str(_dr.bg_noise_dir)
    if hasattr(_dr, "music_dir") and _dr.music_dir and Path(_dr.music_dir).exists():
        _aug_dirs["music_folder"] = str(_dr.music_dir)
    if hasattr(_dr, "rir_dir") and _dr.rir_dir and Path(_dr.rir_dir).exists():
        _aug_dirs["rir_folder"] = str(_dr.rir_dir)

# Split into wakes / NWW pool
wakes    = [(p, l) for p, l in train_rows if str(l) == "1"]
nww_pool = [(p, l) for p, l in train_rows if str(l) == "0"]

# Optionally extend pool with local files
_local_nww_dir = DATASET_DIR / "negatives" / "not_wake_word_subset"
if _local_nww_dir.exists():
    _existing = {p for p, _ in nww_pool}
    _extra = [(str(f), "0") for f in sorted(_local_nww_dir.glob("*.wav"))
              if str(f) not in _existing]
    nww_pool += _extra
    if _extra:
        print(f"Extended NWW pool with {len(_extra)} local files")

print(f"Positives: {len(wakes)}  |  NWW pool: {len(nww_pool)}  |  Test: {len(test_rows)}")
print(f"Augmentation: {list(_aug_dirs.keys()) or 'none'}")

if len(nww_pool) < 50:
    print("WARNING: NWW pool is very small (<50). "
          "Hard-negative mining will be ineffective. "
          "Consider downloading a larger negative corpus.")

## Cell 6 — Smoke test

Runs the infinite loop for a maximum of **5 epochs** with a tiny scan to validate that everything works end-to-end before committing to a full run. Safe to re-run.

Expected outputs:
- At least one `[HardNeg]` log line with counts
- `best_f1.pt`, `best_f1.onnx`, `best_f1_featurizer.onnx` written to `MODEL_DIR/smoke_test/`
- `metrics.csv` with 5 rows
- Exits cleanly with "Best F1: X.XXXX"

If this cell errors, fix before running Cell 7.

In [ ]:
import os
from pathlib import Path
from ww_trainer.trainer import WakeWordTrainer
from ww_trainer.infinite_loop import StoppingGoal, infinite_training_loop

_smoke_dir = MODEL_DIR.parent / f"{ARCH}_{LOSS}_h{HIDDEN_DIM}_smoke_test"
_smoke_dir.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("SMOKE TEST — 5 epochs, scan_size=min(200, pool)")
print("=" * 60)

_smoke_trainer = WakeWordTrainer(
    arch=ARCH,
    featurizer="",
    feature_dim=None,
    featurizer_type="mfcc",
    n_mfcc=40,
    hidden_dim=HIDDEN_DIM,
    dropout=0.1,
    device=DEVICE,
    losses_cfg=[{"name": LOSS, "weight": 1.0}],
    export_onnx=True,
    seed=SEED,
    wake_word=WAKE_WORD,
    mlflow_uri=os.environ.get("MLFLOW_TRACKING_URI"),
    **_aug_dirs,
)

_smoke_goal = StoppingGoal(
    target_f1=0.01,   # trivially easy — exits on patience, not metric
    target_eer=0.99,
    patience_after=1,
    hard_plateau=5,
    min_epochs=3,
    max_epochs=5,
)

_smoke_f1 = infinite_training_loop(
    trainer=_smoke_trainer,
    output_dir=_smoke_dir,
    wakes=wakes,
    nww_pool=nww_pool,
    test_data=test_rows,
    goal=_smoke_goal,
    scan_size=min(200, len(nww_pool)),
    neg_threshold=NEG_THRESHOLD,
    neg_multiplier=NEG_MULTIPLIER,
    lr=LR,
    batch_size=min(BATCH_SIZE, 8),
    aug_prob=0.3,
    use_mixup=False,
    spec_augment=False,
    resume_cache=False,
)

print(f"\nSmoke test complete. Best F1: {_smoke_f1:.4f}")

# Validate outputs
_expected = ["best_f1.pt", "best_f1.onnx", "best_f1_featurizer.onnx", "metrics.csv"]
for fname in _expected:
    p = _smoke_dir / fname
    status = f"OK  ({p.stat().st_size//1024} KB)" if p.exists() else "MISSING"
    print(f"  {fname}: {status}")

## Cell 7 — Full infinite training run

Trains until all goals are met + patience, or until `MAX_EPOCHS` is hit, or until a hard plateau is detected.

The run is **resumable**: `mining_cache.pt` is saved after every epoch. If the kernel restarts, re-run Cells 2–5 then this cell — it will load the cache and continue from where it left off.

**On Kaggle**: sessions time out after ~9 hours. Set `MAX_EPOCHS=500` as a safety ceiling and use the cache to resume across sessions.

In [ ]:
import os
from ww_trainer.trainer import WakeWordTrainer
from ww_trainer.infinite_loop import StoppingGoal, infinite_training_loop

MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer = WakeWordTrainer(
    arch=ARCH,
    featurizer="",
    feature_dim=None,
    featurizer_type="mfcc",
    n_mfcc=40,
    hidden_dim=HIDDEN_DIM,
    dropout=0.1,
    device=DEVICE,
    losses_cfg=[{"name": LOSS, "weight": 1.0}],
    export_onnx=True,
    seed=SEED,
    wake_word=WAKE_WORD,
    mlflow_uri=os.environ.get("MLFLOW_TRACKING_URI"),
    **_aug_dirs,
)

goal = StoppingGoal(
    target_f1=TARGET_F1,
    target_eer=TARGET_EER,
    patience_after=PATIENCE_AFTER,
    hard_plateau=HARD_PLATEAU,
    min_epochs=MIN_EPOCHS,
    max_epochs=MAX_EPOCHS,
)

print(f"Goals: {goal.summary()}")
print(f"NWW pool: {len(nww_pool)} files | scan_size: {SCAN_SIZE}/epoch")
print(f"VC: {VC_PER_EPOCH} positives/epoch" if VC_PER_EPOCH else "VC: disabled")
print()

best_f1 = infinite_training_loop(
    trainer=trainer,
    output_dir=MODEL_DIR,
    wakes=wakes,
    nww_pool=nww_pool,
    test_data=test_rows,
    goal=goal,
    scan_size=SCAN_SIZE,
    neg_threshold=NEG_THRESHOLD,
    neg_multiplier=NEG_MULTIPLIER,
    lr=LR,
    batch_size=BATCH_SIZE,
    aug_prob=0.7,
    use_mixup=True,
    spec_augment=True,
    vc_per_epoch=VC_PER_EPOCH,
    vc_backend=VC_BACKEND,
    resume_cache=True,
)

print(f"\nBest F1: {best_f1:.4f}")
print(f"Model:   {MODEL_DIR}/best_f1.pt")
print(f"ONNX:    {MODEL_DIR}/best_f1.onnx")

## Cell 8 — Per-epoch training dashboard

Plots from `metrics.csv`:
- F1 and EER over epochs
- Hard-negative count per epoch (from MLflow or estimated from log)
- Goal target lines

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

_metrics_csv = MODEL_DIR / "metrics.csv"
if not _metrics_csv.exists():
    print(f"No metrics.csv found at {_metrics_csv} — run Cell 7 first.")
else:
    df = pd.read_csv(_metrics_csv)
    print(f"Loaded {len(df)} epochs from {_metrics_csv}")
    print(df.tail(5).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"Infinite Training — {WAKE_WORD!r}  ({ARCH} / {LOSS})", fontsize=12)

    # F1 curve
    ax = axes[0]
    if "f1" in df.columns:
        ax.plot(df.index + 1, df["f1"], label="F1", color="steelblue")
        ax.axhline(TARGET_F1, color="steelblue", linestyle="--", alpha=0.5, label=f"target F1={TARGET_F1}")
    if "loss" in df.columns:
        ax2 = ax.twinx()
        ax2.plot(df.index + 1, df["loss"], color="salmon", alpha=0.5, label="loss")
        ax2.set_ylabel("Loss", color="salmon")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1.05)
    ax.set_title("F1 over epochs")
    ax.legend(loc="lower right", fontsize=8)

    # EER curve
    ax = axes[1]
    if "eer" in df.columns:
        ax.plot(df.index + 1, df["eer"], label="EER", color="darkorange")
        ax.axhline(TARGET_EER, color="darkorange", linestyle="--", alpha=0.5, label=f"target EER={TARGET_EER}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("EER")
    ax.set_ylim(0, 1.05)
    ax.set_title("EER over epochs (lower = better)")
    ax.legend(loc="upper right", fontsize=8)

    plt.tight_layout()
    _plot_path = str(MODEL_DIR / "training_curve.png")
    plt.savefig(_plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {_plot_path}")

## Cell 9 — NWW pool size ablation

Trains the same model with four NWW pool sizes (10%, 25%, 50%, 100% of the available pool) for `max_epochs=15` each and compares best F1. This answers: *"how much does a larger negative pool actually help?"*

Skip this cell if `len(nww_pool) < 100` — the ablation is only meaningful with a decent pool.

In [ ]:
import json, random, os
from pathlib import Path
from ww_trainer.trainer import WakeWordTrainer
from ww_trainer.infinite_loop import StoppingGoal, infinite_training_loop

if len(nww_pool) < 100:
    print(f"NWW pool too small ({len(nww_pool)} samples) for a meaningful ablation — skipping.")
else:
    _fracs = [0.1, 0.25, 0.5, 1.0]
    _ablation_goal = StoppingGoal(
        target_f1=0.01, target_eer=0.99,   # exit via max_epochs
        patience_after=1, hard_plateau=15,
        min_epochs=5, max_epochs=15,
    )
    _ablation_results = []
    random.seed(SEED)

    for frac in _fracs:
        n = max(10, int(len(nww_pool) * frac))
        pool_subset = random.sample(nww_pool, n)
        _abl_dir = MODEL_DIR.parent / f"{ARCH}_{LOSS}_h{HIDDEN_DIM}_ablation_pool{int(frac*100)}"
        _result_file = _abl_dir / "ablation_result.json"

        if _result_file.exists():
            saved = json.loads(_result_file.read_text())
            print(f"SKIP pool={n} ({frac:.0%}) — already done: F1={saved['best_f1']:.4f}")
            _ablation_results.append(saved)
            continue

        print(f"\nAblation: pool={n} ({frac:.0%} of {len(nww_pool)})")
        _abl_dir.mkdir(parents=True, exist_ok=True)

        _abl_trainer = WakeWordTrainer(
            arch=ARCH, featurizer="", feature_dim=None,
            featurizer_type="mfcc", n_mfcc=40, hidden_dim=HIDDEN_DIM,
            dropout=0.1, device=DEVICE,
            losses_cfg=[{"name": LOSS, "weight": 1.0}],
            export_onnx=False, seed=SEED, wake_word=WAKE_WORD,
            mlflow_uri=None, **_aug_dirs,
        )

        _abl_f1 = infinite_training_loop(
            trainer=_abl_trainer, output_dir=_abl_dir,
            wakes=wakes, nww_pool=pool_subset, test_data=test_rows,
            goal=_ablation_goal,
            scan_size=min(200, len(pool_subset)),
            neg_threshold=NEG_THRESHOLD, neg_multiplier=NEG_MULTIPLIER,
            lr=LR, batch_size=min(BATCH_SIZE, 8),
            aug_prob=0.3, use_mixup=False, spec_augment=False, resume_cache=False,
        )
        row = {"frac": frac, "pool_size": n, "best_f1": _abl_f1}
        _result_file.write_text(json.dumps(row))
        _ablation_results.append(row)
        print(f"  pool={n}: best F1={_abl_f1:.4f}")

    # Plot
    import matplotlib.pyplot as plt
    import pandas as pd
    _adf = pd.DataFrame(_ablation_results).sort_values("pool_size")
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(_adf["pool_size"], _adf["best_f1"], "o-", color="steelblue")
    ax.set_xlabel("NWW pool size")
    ax.set_ylabel("Best F1 (15 epochs)")
    ax.set_title(f"NWW pool size vs F1 — {WAKE_WORD!r}")
    ax.set_ylim(0, 1.05)
    for _, row in _adf.iterrows():
        ax.annotate(f"{row['best_f1']:.3f}", (row["pool_size"], row["best_f1"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=8)
    plt.tight_layout()
    _plot_path = str(MODEL_DIR.parent / "pool_ablation.png")
    plt.savefig(_plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {_plot_path}")
    print(_adf[["pool_size", "frac", "best_f1"]].to_string(index=False))

## Cell 10 — ONNX verification & inference test

Confirms both ONNX files are present and runs a confidence score on a positive sample from the test set.

In [ ]:
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

_head_onnx = MODEL_DIR / "best_f1.onnx"
_feat_onnx = MODEL_DIR / "best_f1_featurizer.onnx"

print("ONNX files:")
for label, path in [("featurizer", _feat_onnx), ("classifier", _head_onnx)]:
    if path.exists():
        print(f"  OK  {label}: {path.name} ({path.stat().st_size//1024} KB)")
    else:
        print(f"  MISSING  {label}: {path}")

if _head_onnx.exists() and _feat_onnx.exists():
    inferencer = OnnxWakeWordInferencer(str(_feat_onnx), str(_head_onnx))

    # Find a positive sample from test set
    _pos_path = None
    with open(test_rows[0][0] if not hasattr(test_rows, '__fspath__') else test_rows,
              errors='ignore') as _f:
        pass  # test_rows is already a list
    for _p, _l in test_rows:
        if str(_l) == "1" and Path(_p).exists():
            _pos_path = _p
            break

    if _pos_path:
        _wav, _sr = torchaudio.load(_pos_path)
        if _sr != 16000:
            _wav = torchaudio.functional.resample(_wav, _sr, 16000)
        _wav_np = _wav.mean(0).numpy().astype(np.float32)
        _score = inferencer.infer(_wav_np)
        print(f"\nInference on: {Path(_pos_path).name}")
        print(f"  Confidence: {_score:.4f}  "
              f"({'PASS ✓' if _score > 0.5 else 'LOW — may need more training'})")
    else:
        print("\nNo positive test sample found — skipping inference check.")

    print(f"\nTo test on a WAV file locally:")
    print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
    print(f"      --featurizer {_feat_onnx} \\")
    print(f"      --model      {_head_onnx} \\")
    print(f"      --audio      sample.wav")